# Code Ownership, notebook 5: model comparison and the choice

Rules fixed in the protocol: (1) a head more complex than ridge is kept only if it beats the best ridge by a paired grouped-bootstrap interval that excludes 0 and wins in
at least 4 of 5 folds; (2) the encoder model is used in production only if it beats the best hand-built-feature baseline by an interval that excludes 0, otherwise production
uses the hand-feature head (rule added before the final test, see the amendment log).

In [1]:
import sys, warnings, json, pickle
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("../../helpers").resolve()))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import spearmanr
import ko_data as K, ko_models as M
from core import metrics, group_bootstrap, paired_diff_bootstrap, qwk, band
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 90)
RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)
d = K.load()
dev = d[d.role == "dev"].reset_index(drop=True)
test = d[d.role == "test"].reset_index(drop=True)
y, groups, folds = dev.score.to_numpy(), dev.snippet_id.to_numpy(), dev.fold.to_numpy()
def ci(p, stat):
    return group_bootstrap(y, p, groups, stat, n=1000)
def sp(a, b):
    return spearmanr(a, b).correlation
def mae(a, b):
    return float(np.mean(np.abs(a - b)))
def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

In [2]:
P = pickle.load(open(RESULTS / "tuning_preds.pkl", "rb")); B = pickle.load(open(RESULTS / "baselines_oof.pkl", "rb"))
def fold_wins(p_a, p_b):
    return int(sum(mae(p_a[folds == f], y[folds == f]) < mae(p_b[folds == f], y[folds == f]) for f in sorted(set(folds))))
rows = []
for key, v in P.items():
    e, s, h = key.split("|"); m = metrics(y, v["pred"])
    rows.append({"key": key, "encoder": e, "set": s, "head": h, "MAE": m["MAE"], "RMSE": m["RMSE"], "Spearman": m["spearman"], "QWK": m["band_qwk"]})
T = pd.DataFrame(rows)
best_ridge = T[T["head"] == "ridge"].sort_values("MAE").iloc[0]
best_complex = T[T["head"] != "ridge"].sort_values("MAE").iloc[0]
print("best ridge:   ", best_ridge.key, round(best_ridge.MAE, 3))
print("best complex: ", best_complex.key, round(best_complex.MAE, 3))

best ridge:    codebert|C+H|ridge 0.197
best complex:  codebert|B+H|svr 0.161


In [3]:
# rule 1: complexity
pr, pc = P[best_ridge.key]["pred"], P[best_complex.key]["pred"]
lo, hi = paired_diff_bootstrap(y, pr, pc, groups, n=2000)
wins = fold_wins(pc, pr)
keep_complex = bool(lo > 0 and wins >= 4)
print(f"MAE(ridge) - MAE(complex): 95% interval [{lo:.3f}, {hi:.3f}]; complex wins {wins} of 5 folds -> keep complex head: {keep_complex}")
chosen = best_complex if keep_complex else best_ridge
chosen_pred = P[chosen.key]["pred"]
print("chosen encoder model:", chosen.key)

MAE(ridge) - MAE(complex): 95% interval [0.017, 0.054]; complex wins 5 of 5 folds -> keep complex head: True
chosen encoder model: codebert|B+H|svr


In [4]:
# rule 2: does the encoder model beat the best hand-built-feature baseline?
hand = {k: v for k, v in B.items() if k.startswith("hand-built")}
best_hand = min(hand, key=lambda k: mae(hand[k], y))
lo2, hi2 = paired_diff_bootstrap(y, hand[best_hand], chosen_pred, groups, n=2000)
wins2 = fold_wins(chosen_pred, hand[best_hand])
encoder_helps = bool(lo2 > 0)
print(f"best hand-feature baseline: {best_hand} (MAE {mae(hand[best_hand], y):.3f}); chosen encoder model MAE {mae(chosen_pred, y):.3f}")
print(f"MAE(hand) - MAE(encoder model): 95% interval [{lo2:.3f}, {hi2:.3f}]; encoder model wins {wins2} of 5 folds -> encoder helps: {encoder_helps}")

best hand-feature baseline: hand-built features, forest (MAE 0.206); chosen encoder model MAE 0.161
MAE(hand) - MAE(encoder model): 95% interval [0.018, 0.071]; encoder model wins 5 of 5 folds -> encoder helps: True


In [5]:
# comparison table: baselines and the best of each encoder/head
tab = []
for name, p in B.items():
    m = metrics(y, p); tab.append({"model": name, "kind": "baseline", "MAE": m["MAE"], "RMSE": m["RMSE"], "Spearman": m["spearman"], "QWK": m["band_qwk"]})
for e in sorted(T.encoder.unique()):
    for h in ("ridge", "svr", "kernel_ridge", "forest", "boosting", "mlp"):
        r = T[(T.encoder == e) & (T["head"] == h)].sort_values("MAE").iloc[0]
        tab.append({"model": f"{e} / {r.set} / {h}", "kind": "encoder", "MAE": r.MAE, "RMSE": r.RMSE, "Spearman": r.Spearman, "QWK": r.QWK})
tab = pd.DataFrame(tab).sort_values("MAE").round(3)
tab.to_csv(RESULTS / "comparison.csv", index=False)
tab

,model,kind,MAE,RMSE,Spearman,QWK
10,codebert / B+H / svr,encoder,0.161,0.216,0.812,0.787
13,codebert / A+H / boosting,encoder,0.174,0.225,0.795,0.752
12,codebert / B+H / forest,encoder,0.175,0.230,0.796,0.750
22,unixcoder / A+H / svr,encoder,0.187,0.243,0.746,0.684
16,mpnet / A+H / svr,encoder,0.192,0.250,0.728,0.684
9,codebert / C+H / ridge,encoder,0.197,0.251,0.745,0.666
7,"hand-built features, forest",baseline,0.206,0.274,0.649,0.585
11,codebert / A+H / kernel_ridge,encoder,0.206,0.254,0.751,0.642
18,mpnet / A+H / forest,encoder,0.208,0.271,0.648,0.631
25,unixcoder / D+H / boosting,encoder,0.212,0.269,0.668,0.603


In [6]:
# intervals for the main candidates
cands = {"training mean": B["training mean"], "answer length (words)": B["answer length (words)"], best_hand: hand[best_hand], chosen.key: chosen_pred}
for name, p in cands.items():
    print(f"{name:45s} MAE {mae(p, y):.3f} {np.round(ci(p, mae), 3)}   Spearman {sp(p, y):.3f} {np.round(ci(p, lambda a, b: sp(a, b)), 3)}")

training mean                                 MAE 0.319 [0.309 0.328]   Spearman -0.025 [-0.048 -0.004]


answer length (words)                         MAE 0.309 [0.289 0.326]   Spearman 0.446 [0.355 0.573]


hand-built features, forest                   MAE 0.206 [0.185 0.228]   Spearman 0.649 [0.569 0.73 ]


codebert|B+H|svr                              MAE 0.161 [0.143 0.182]   Spearman 0.812 [0.754 0.865]


In [7]:
if encoder_helps:
    production = {"kind": "encoder", "key": chosen.key, "encoder": chosen.encoder, "feature_set": chosen.set, "head": chosen["head"]}
else:
    production = {"kind": "hand", "key": best_hand, "encoder": None, "feature_set": "H", "head": best_hand.split(", ")[1]}
choice = {"production": production, "encoder_model": chosen.key, "keep_complex_head": keep_complex, "complex_vs_ridge_interval": [lo, hi], "complex_folds_won": wins,
          "encoder_helps": encoder_helps, "encoder_vs_hand_interval": [lo2, hi2], "encoder_folds_won": wins2, "best_hand_baseline": best_hand,
          "dev_MAE": {"production": mae(chosen_pred if encoder_helps else hand[best_hand], y), "mean": mae(B["training mean"], y), "length": mae(B["answer length (words)"], y)}}
(RESULTS / "model_choice.json").write_text(json.dumps(choice, indent=1, default=float), encoding="utf-8")
choice["production"]

{'kind': 'encoder',
 'key': 'codebert|B+H|svr',
 'encoder': 'codebert',
 'feature_set': 'B+H',
 'head': 'svr'}

## Reading the results

**The two choice rules, as fixed in the protocol.** Best ridge: CodeBERT / answer + similarities + hand features / ridge, MAE 0.197. Best complex head: CodeBERT / answer + similarities + hand features / SVR, MAE 0.161. The SVR beats the ridge by 0.017 to 0.054 MAE (95% interval over functions, entirely above 0) and wins in 5 of 5 folds, so the complex head is kept. Against the best hand-built-feature baseline (forest, MAE 0.206) the chosen model is better by 0.018 to 0.071 and wins 5 of 5 folds, so the encoder is used in production. Development out-of-fold performance of the chosen model: MAE 0.161 (95% 0.143 to 0.182), Spearman 0.81 (0.75 to 0.87), QWK 0.79.

**Caution.** The model was chosen as the best of 144 combinations on these same folds. The tuning of settings is nested and free of selection bias, but picking the best combination is a mild selection effect; the final test in notebook 6 is the unbiased check. Mean prediction has MAE 0.319 and answer length 0.309.
